# Fundamentos de PyTorch

PyTorch es el framework de deep learning más popular en investigación y cada vez más en producción. Este notebook cubre los conceptos fundamentales que necesitas para construir, entrenar y desplegar modelos.

**Contenido:**
1. Tensores: la estructura de datos fundamental
2. Autograd: diferenciación automática
3. nn.Module: construir modelos
4. Dataset y DataLoader: gestión de datos
5. Training loop completo
6. Guardar y cargar modelos
7. Ejemplo práctico: clasificación con Iris

**Requisitos:**
```bash
pip install torch scikit-learn matplotlib
```

## Tensores

Los **tensores** son la estructura de datos fundamental en PyTorch, equivalentes a los arrays de NumPy pero con dos superpoderes:

1. **Aceleración GPU**: se pueden mover a GPU para cálculos paralelos masivos
2. **Autograd**: pueden rastrear operaciones para calcular gradientes automáticamente

| NumPy | PyTorch | Descripción |
|-------|---------|-------------|
| `np.array` | `torch.tensor` | Crear tensor |
| `np.zeros` | `torch.zeros` | Tensor de ceros |
| `array.shape` | `tensor.shape` | Dimensiones |
| `array.dtype` | `tensor.dtype` | Tipo de datos |
| `array.reshape` | `tensor.view`/`reshape` | Cambiar forma |

In [ ]:
import torch
import numpy as np

# --- Device detection (CUDA for NVIDIA, MPS for Apple Silicon, CPU fallback) ---
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"Using CUDA GPU: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
    print("Using Apple Silicon GPU (MPS)")
else:
    device = torch.device('cpu')
    print("Using CPU")

print(f"PyTorch version: {torch.__version__}")

# --- Creating tensors ---
# From Python list
a = torch.tensor([1.0, 2.0, 3.0])
print(f"\nTensor a: {a}, dtype: {a.dtype}, device: {a.device}")

# Common initializations
zeros = torch.zeros(3, 4)
ones = torch.ones(2, 3, dtype=torch.float32)
rand = torch.randn(3, 3)         # Standard normal
arange = torch.arange(0, 10, 2)  # Like np.arange

print(f"Zeros shape: {zeros.shape}")
print(f"Random 3x3:\n{rand}")

# --- NumPy <-> PyTorch conversion ---
np_array = np.array([1.0, 2.0, 3.0])
tensor_from_np = torch.from_numpy(np_array)  # Shares memory!
back_to_np = tensor_from_np.numpy()           # Shares memory!
print(f"\nNumPy -> Tensor: {tensor_from_np}")

# --- Operations ---
x = torch.randn(3, 4)
y = torch.randn(3, 4)
print(f"\nElement-wise add: {(x + y).shape}")
print(f"Matrix multiply (3,4) @ (4,2): {(x @ torch.randn(4, 2)).shape}")
print(f"Sum all: {x.sum():.4f}")
print(f"Mean per column: {x.mean(dim=0)}")

# --- Reshaping ---
t = torch.arange(12)
print(f"\nOriginal: {t.shape}")
print(f"View (3,4): {t.view(3, 4).shape}")
print(f"Unsqueeze(0): {t.unsqueeze(0).shape}")   # Add batch dim: (1, 12)
print(f"Unsqueeze(1): {t.view(3, 4).unsqueeze(1).shape}")  # (3, 1, 4)

# --- GPU transfer ---
t_gpu = t.float().to(device)
print(f"\nTensor on {t_gpu.device}: {t_gpu[:5]}")

# Move back to CPU (required for NumPy conversion)
t_cpu = t_gpu.cpu()
print(f"Back on CPU: {t_cpu.device}")

## Autograd: diferenciación automática

**Autograd** es el motor que hace posible entrenar redes neuronales. Rastrea todas las operaciones sobre tensores y calcula gradientes automáticamente mediante la regla de la cadena (backpropagation).

**Concepto clave:**
- `requires_grad=True`: indica que queremos calcular gradientes respecto a este tensor
- `.backward()`: calcula los gradientes (backpropagation)
- `.grad`: contiene el gradiente acumulado

Para una función `L = f(w)`, autograd calcula `dL/dw` automáticamente.

In [ ]:
import torch

# --- Simple gradient computation ---
# f(x) = x^2 + 3x + 1, df/dx = 2x + 3
x = torch.tensor(2.0, requires_grad=True)
f = x**2 + 3*x + 1
f.backward()

print(f"x = {x.item()}")
print(f"f(x) = {f.item()}")
print(f"df/dx = {x.grad.item()} (expected: {2*x.item() + 3})")

# --- Gradient of a vector function ---
# Simulating a simple neural network forward pass
w = torch.randn(3, requires_grad=True)
b = torch.tensor(0.5, requires_grad=True)
x_input = torch.tensor([1.0, 2.0, 3.0])
y_true = torch.tensor(1.0)

# Forward pass: prediction = dot(w, x) + b
y_pred = torch.dot(w, x_input) + b

# Loss: MSE
loss = (y_pred - y_true) ** 2

# Backward pass
loss.backward()

print(f"\n--- Neural network gradient example ---")
print(f"Weights: {w.data}")
print(f"Prediction: {y_pred.item():.4f}")
print(f"Loss: {loss.item():.4f}")
print(f"dL/dw: {w.grad}")
print(f"dL/db: {b.grad.item():.4f}")

# --- Manual gradient descent step ---
lr = 0.01
with torch.no_grad():  # Don't track this operation
    w_new = w - lr * w.grad
    b_new = b - lr * b.grad

print(f"\nAfter 1 gradient step (lr={lr}):")
print(f"w: {w.data} -> {w_new}")
print(f"b: {b.item():.4f} -> {b_new.item():.4f}")

# IMPORTANT: zero gradients before next backward pass
# (gradients accumulate by default in PyTorch)
print(f"\nGrad before zero: {w.grad}")
w.grad.zero_()
print(f"Grad after zero:  {w.grad}")

## nn.Module: construir modelos

`torch.nn.Module` es la clase base para todos los modelos en PyTorch. Para crear un modelo:

1. Heredar de `nn.Module`
2. Definir las capas en `__init__`
3. Definir el forward pass en `forward()`

PyTorch calcula el backward pass automáticamente gracias a autograd. Solo necesitamos definir cómo fluyen los datos hacia adelante.

In [ ]:
import torch
import torch.nn as nn

class SimpleNet(nn.Module):
    """Simple neural network with 2 hidden layers and ReLU activation."""
    
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)
        self.output = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.dropout(x)
        x = self.relu(self.layer2(x))
        x = self.dropout(x)
        x = self.output(x)
        return x

# Create model
model = SimpleNet(input_dim=10, hidden_dim=64, output_dim=3)
print("=== Model Architecture ===")
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Forward pass with dummy data
batch = torch.randn(5, 10)  # 5 samples, 10 features
output = model(batch)
print(f"\nInput shape:  {batch.shape}")
print(f"Output shape: {output.shape}")
print(f"Output (logits):\n{output.data}")

# Using nn.Sequential for simpler architectures
simple_model = nn.Sequential(
    nn.Linear(10, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 3)
)
print(f"\n=== Sequential Model ===")
print(simple_model)

## Dataset y DataLoader

PyTorch proporciona dos abstracciones para gestionar datos eficientemente:

- **`Dataset`**: define cómo acceder a un ejemplo individual (implementar `__len__` y `__getitem__`)
- **`DataLoader`**: itera sobre el Dataset en mini-batches con shuffling, paralelismo, etc.

Esto permite:
- Cargar datos que no caben en memoria (lazy loading)
- Aplicar transformaciones on-the-fly
- Paralelizar la carga de datos con múltiples workers

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class CustomDataset(Dataset):
    """Custom dataset example for tabular data."""
    
    def __init__(self, X, y, transform=None):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
        self.transform = transform
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        sample = self.X[idx]
        label = self.y[idx]
        
        if self.transform:
            sample = self.transform(sample)
        
        return sample, label

# Create synthetic data
np.random.seed(42)
X_data = np.random.randn(1000, 10).astype(np.float32)
y_data = np.random.randint(0, 3, 1000)

# Create Dataset and DataLoader
dataset = CustomDataset(X_data, y_data)

train_loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,       # Shuffle training data each epoch
    num_workers=0,      # Number of parallel data loading processes
    drop_last=True      # Drop last incomplete batch
)

print(f"Dataset size: {len(dataset)}")
print(f"Number of batches: {len(train_loader)}")

# Iterate over one batch
for batch_X, batch_y in train_loader:
    print(f"\nBatch X shape: {batch_X.shape}")
    print(f"Batch y shape: {batch_y.shape}")
    print(f"Batch y values: {batch_y[:10]}")
    break  # Just show one batch

# Using TensorDataset for simple cases
from torch.utils.data import TensorDataset

simple_dataset = TensorDataset(
    torch.FloatTensor(X_data),
    torch.LongTensor(y_data)
)
simple_loader = DataLoader(simple_dataset, batch_size=64, shuffle=True)
print(f"\nSimple loader batches: {len(simple_loader)}")

## Training Loop completo

El training loop en PyTorch sigue siempre el mismo patrón:

```python
for epoch in range(n_epochs):
    model.train()              # Modo entrenamiento (activa dropout, batchnorm)
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()  # 1. Limpiar gradientes
        output = model(batch_X)# 2. Forward pass
        loss = criterion(output, batch_y)  # 3. Calcular loss
        loss.backward()        # 4. Backward pass (calcular gradientes)
        optimizer.step()       # 5. Actualizar pesos
    
    model.eval()               # Modo evaluación
    with torch.no_grad():      # No calcular gradientes
        # Evaluar en validation set
```

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

# --- Generate synthetic classification data ---
np.random.seed(42)
torch.manual_seed(42)

n_samples = 2000
n_features = 20
n_classes = 4

# Create separable clusters
X_all = []
y_all = []
for i in range(n_classes):
    center = np.random.randn(n_features) * 3
    X_class = np.random.randn(n_samples // n_classes, n_features) + center
    X_all.append(X_class)
    y_all.append(np.full(n_samples // n_classes, i))

X_all = np.vstack(X_all).astype(np.float32)
y_all = np.concatenate(y_all)

# Shuffle and split
indices = np.random.permutation(n_samples)
X_all, y_all = X_all[indices], y_all[indices]

split = int(0.8 * n_samples)
X_train_t = torch.FloatTensor(X_all[:split])
y_train_t = torch.LongTensor(y_all[:split])
X_val_t = torch.FloatTensor(X_all[split:])
y_val_t = torch.LongTensor(y_all[split:])

# DataLoaders
train_ds = TensorDataset(X_train_t, y_train_t)
val_ds = TensorDataset(X_val_t, y_val_t)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128)

# --- Model ---
model = nn.Sequential(
    nn.Linear(n_features, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(64, n_classes)
)

# --- Training setup ---
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

# --- Training loop ---
n_epochs = 50
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(n_epochs):
    # Training phase
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * batch_X.size(0)
        _, predicted = outputs.max(1)
        train_total += batch_y.size(0)
        train_correct += predicted.eq(batch_y).sum().item()
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            
            val_loss += loss.item() * batch_X.size(0)
            _, predicted = outputs.max(1)
            val_total += batch_y.size(0)
            val_correct += predicted.eq(batch_y).sum().item()
    
    # Calculate epoch metrics
    train_loss /= train_total
    val_loss /= val_total
    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    
    scheduler.step(val_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{n_epochs} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

print(f"\nFinal validation accuracy: {history['val_acc'][-1]:.4f}")

## Guardar y cargar modelos

PyTorch ofrece dos formas de guardar modelos:

1. **`state_dict` (recomendado)**: guarda solo los pesos. Requiere tener la definición de la clase para reconstruir.
2. **Modelo completo**: guarda arquitectura + pesos. Más frágil (depende de la estructura de archivos).

Siempre recordar poner el modelo en modo `eval()` antes de inferencia para desactivar dropout y batchnorm de entrenamiento.

In [ ]:
import torch

# --- Save model (recommended: state_dict) ---
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': n_epochs,
    'history': history
}, 'model_checkpoint.pt')
print("Model checkpoint saved.")

# --- Load model ---
# Recreate model architecture
loaded_model = nn.Sequential(
    nn.Linear(n_features, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(64, n_classes)
)

checkpoint = torch.load('model_checkpoint.pt', weights_only=False)
loaded_model.load_state_dict(checkpoint['model_state_dict'])
print(f"Model loaded from epoch {checkpoint['epoch']}")

# Set to evaluation mode (disables dropout, changes batchnorm behavior)
loaded_model.eval()

# Verify: predictions should match
with torch.no_grad():
    original_pred = model(X_val_t[:5])
    loaded_pred = loaded_model(X_val_t[:5])

# Note: model must also be in eval mode for exact match
model.eval()
with torch.no_grad():
    original_pred = model(X_val_t[:5])
    loaded_pred = loaded_model(X_val_t[:5])

print(f"Predictions match: {torch.allclose(original_pred, loaded_pred)}")

## Ejemplo práctico: clasificación con datos reales

Vamos a aplicar todo lo aprendido al dataset **Iris** de scikit-learn. Es un dataset pequeño pero perfecto para verificar que nuestro pipeline funciona correctamente.

**Pasos:**
1. Cargar datos con scikit-learn
2. Convertir a tensores de PyTorch
3. Entrenar una red neuronal
4. Visualizar la curva de loss
5. Evaluar el modelo

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np

# Load and prepare data
iris = load_iris()
X, y = iris.data, iris.target

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert to tensors
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.LongTensor(y_train)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.LongTensor(y_test)

# DataLoaders
train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)

# Model
class IrisNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 3)
        )
    
    def forward(self, x):
        return self.net(x)

torch.manual_seed(42)
model = IrisNet()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training
epochs = 100
train_losses = []
test_losses = []

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        output = model(batch_X)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    train_losses.append(epoch_loss / len(train_loader))
    
    model.eval()
    with torch.no_grad():
        test_output = model(X_test_t)
        test_loss = criterion(test_output, y_test_t)
        test_losses.append(test_loss.item())

# Plot loss curves
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_losses, label='Train Loss', linewidth=2)
ax.plot(test_losses, label='Test Loss', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Curvas de entrenamiento - Iris Dataset')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, classification_report
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

# --- Evaluate ---
model.eval()
with torch.no_grad():
    logits = model(X_test_t)
    predictions = logits.argmax(dim=1).numpy()

print("=== Test Results ===")
print(f"Accuracy: {accuracy_score(y_test, predictions):.4f}")
print(f"\n{classification_report(y_test, predictions, target_names=iris.target_names)}")

# --- Visualize decision boundary (2D PCA projection) ---
pca = PCA(n_components=2)
X_all_2d = pca.fit_transform(np.vstack([X_train, X_test]))

# Create a mesh grid for the decision boundary
x_min, x_max = X_all_2d[:, 0].min() - 1, X_all_2d[:, 0].max() + 1
y_min, y_max = X_all_2d[:, 1].min() - 1, X_all_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                       np.linspace(y_min, y_max, 200))

# Transform mesh points back to original space for prediction
mesh_points_2d = np.c_[xx.ravel(), yy.ravel()]
mesh_points_original = pca.inverse_transform(mesh_points_2d)
mesh_tensor = torch.FloatTensor(mesh_points_original)

model.eval()
with torch.no_grad():
    mesh_pred = model(mesh_tensor).argmax(dim=1).numpy()

Z = mesh_pred.reshape(xx.shape)

fig, ax = plt.subplots(figsize=(10, 7))
ax.contourf(xx, yy, Z, alpha=0.3, cmap='Set3')

# Plot test points
X_test_2d = pca.transform(X_test)
colors = ['#e74c3c', '#2ecc71', '#3498db']
for i, name in enumerate(iris.target_names):
    mask = y_test == i
    ax.scatter(X_test_2d[mask, 0], X_test_2d[mask, 1],
              c=colors[i], label=name, edgecolors='black', s=60)

ax.set_title('Decision Boundary (2D PCA projection)', fontsize=13)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend()
plt.tight_layout()
plt.show()

## Resumen: template de proyecto PyTorch

Todo proyecto de PyTorch sigue esta estructura:

```
project/
  data/           # Datasets
  models/         # Definiciones de modelos (nn.Module)
  utils/          # Funciones auxiliares
  train.py        # Script de entrenamiento
  evaluate.py     # Script de evaluación
  config.yaml     # Hiperparámetros
```

**Checklist de cada proyecto:**
1. Detectar dispositivo (CUDA/MPS/CPU) y mover modelo + datos
2. Crear Dataset y DataLoader personalizados
3. Definir modelo heredando de `nn.Module`
4. Elegir loss function y optimizer adecuados
5. Implementar training loop con validación
6. Guardar checkpoints regularmente
7. Monitorear métricas con TensorBoard o Weights & Biases
8. Evaluar en test set solo al final

**Siguientes pasos:**
- CNNs para visión (siguiente notebook)
- RNNs/Transformers para NLP
- Transfer Learning para aprovechar modelos preentrenados